<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Lab_1_Cleaner_Robot_SE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 - Städrobot

I den här laborationen ska du utforska kunskapsbaserade AI-agenter i städrobot-domänen.

**Gör labbarna uppifrån och ned.**
<br />
**Du behöver inte läsa längre fram än den del du jobbar med just nu.**

<br />
<hr />

### **Upphovspersoner**
Originalversion: Mattias Tiger, mattias.tiger@liu.se

Förenklad version på svenska: baserad på originalverket ovan (förenklad för nybörjare, ca 13 år).
Anpassning: numpy och matplotlib ersatta med ren Python, all text översatt till svenska.

### **Licens**
CC BY-NC-SA 4.0 <br />
https://creativecommons.org/licenses/by-nc-sa/4.0/

Laborationen (Notebook: kod och instruktioner) är fri att använda, dela, förändra och göra egna versioner av. Det enda villkoret för denna version och alla nya versioner är att 1) samma licens (CC BY-NC-SA 4.0) tillämpas, 2) att ovanstående upphovsperson(er) anges som upphovspersoner till originalversionen och 3) att den inte används i ett sammanhang där en person eller ett företag måste betala för att få tillgång till eller använda versionen. Det vill säga att full tillåtelse ges att använda laborationen, och efterföljande versioner, i skolor eller studiecirklar.
<hr />

# Städrobot - Lab 1.0 - Rutnätsvärlden
Vi börjar med att sätta upp miljön: **Rutnätsvärlden**.

Koden nedan skapar en slumpmässig **miljö** (baserat på önskad täthet av väggar och smuts) för en given rutnätsstorlek.

Vi inkluderar också funktionalitet för att skapa ett **scenario** för en given värld: Placera roboten på en bestämd startplats (dess hem).

Med ett genererat scenario har vi en **probleminstans**. Vi kan sedan ställa frågan: Hur ska roboten röra sig i världen för att städa upp all smuts?

### Steg 1: Skapa en probleminstans

* **Uppgift 1.0.1:** Läs snabbt igenom koden i alla kodceller. Fokusera på klass-/funktionsnamn och beskrivningar (kommentarer).

  * Fråga 1: Världen i domänen består av ett rutnät, där varje cell kan vara av flera olika typer. Hur många celltyper finns det och vilka är de?

* **Uppgift 1.0.2:** Kör kodcellerna.

In [ ]:
import ipywidgets as widgets           # Verktyg för att skapa knappar och gränssnitt
from IPython.display import display, clear_output  # Verktyg för att visa och rensa utskrifter
import random                           # Slumptalsgenerator
import math                             # Matematikfunktioner (t.ex. math.ceil för avrundning uppåt)

# Skapa ett utdatawidget för att visa programutskrifter
# Det används för att fånga print()-anrop inuti funktioner
# Användning:
#   with output:
#     print("Hej Världen")
output = widgets.Output()

In [ ]:
# Funktioner för att skapa och skriva ut rutnätsscenarier
# ---------------------------------------------------------

# Typer av celler i rutnätet
class CellType:
  EMPTY   = 0  # Tom cell
  WALL    = 1  # Vägg
  DIRT    = 2  # Smuts
  HOME    = 3  # Robotens hem (startposition)
  UNKNOWN = 4  # Okänd cell (roboten har inte besökt den ännu)


# Omvandla celltyp till ett tecken för utskrift
def cell_to_string(cell):
  if cell == CellType.EMPTY:
    return "."    # Tom
  if cell == CellType.WALL:
    return "#"    # Vägg
  if cell == CellType.DIRT:
    return "░"    # Smuts
  if cell == CellType.HOME:
    return "H"    # Hem
  if cell == CellType.UNKNOWN:
    return "?"    # Okänd
  return f"Fel, ingen cell kallas {cell}"


def generate_world(grid_size, wall_density, dirt_density):
  # Beräkna den inre ytan (utan kantväggarna)
  area = (grid_size - 2) ** 2

  # Skapa ett tomt rutnät som en lista av listor (rader och kolumner)
  # grid[y][x] - y är raden (uppifrån), x är kolumnen (från vänster)
  grid = [[CellType.EMPTY] * grid_size for _ in range(grid_size)]

  # Lägg till omgivande väggar längs alla kanter
  for row in range(grid_size):
      grid[row][0]             = CellType.WALL  # Vänsterkant
      grid[row][grid_size - 1] = CellType.WALL  # Högerkant
  for col in range(grid_size):
      grid[0][col]             = CellType.WALL  # Överkant
      grid[grid_size - 1][col] = CellType.WALL  # Nederkant

  # Lägg till slumpmässiga väggar inuti rutnätet
  num_walls = math.ceil(area * wall_density)  # Beräkna antal väggar
  for _ in range(num_walls):
    row = 1   # Startposition (uppdateras i loopen)
    col = 1
    tries = 1000  # Max antal försök att hitta en tom cell
    while grid[row][col] != CellType.EMPTY and tries > 0:
      row = random.randint(1, grid_size - 2)  # Slumpmässig rad
      col = random.randint(1, grid_size - 2)  # Slumpmässig kolumn
      tries -= 1
    if tries > 0:
      grid[row][col] = CellType.WALL  # Placera väggen

  # Lägg till slumpmässig smuts inuti rutnätet
  num_dirt = math.ceil(area * dirt_density)  # Beräkna antal smutsceller
  for _ in range(num_dirt):
    row = 1
    col = 1
    tries = 1000
    while grid[row][col] != CellType.EMPTY and tries > 0:
      row = random.randint(1, grid_size - 2)
      col = random.randint(1, grid_size - 2)
      tries -= 1
    if tries > 0:
      grid[row][col] = CellType.DIRT  # Placera smutsen

  return grid


def generate_scenario(grid_size, wall_density, dirt_density, random_home):
  # Generera världen
  grid = generate_world(grid_size, wall_density, dirt_density)

  # Placera roboten vid position (1,1) - övre vänstra hörnet innanför väggen
  robot_pos = (1, 1)  # Startposition (x, y)
  # OBS: grid[y][x] - y-index (rad) kommer först, sedan x-index (kolumn)
  grid[robot_pos[1]][robot_pos[0]] = CellType.HOME  # Markera robotens hem

  return grid, robot_pos


# Hjälpfunktion för att skriva ut rutnätet
def print_grid(grid, robot_location_x, robot_location_y, margin=""):
  for y, row in enumerate(grid):         # Loopa igenom varje rad (y-axel)
      for x, cell in enumerate(row):     # Loopa igenom varje cell i raden (x-axel)
        if x == 0:
          print(margin, end="")          # Skriv ut marginal i början av varje rad
        if robot_location_y == y and robot_location_x == x:
          print("R", end=" ")            # Skriv ut R för robotens position
        else:
          print(cell_to_string(cell), end=" ")  # Skriv ut celltypen
      print()  # Ny rad efter varje rad i rutnätet


# Hjälpfunktion för att skriva ut rutnätet med koordinater
def print_grid_indices(grid, robot_location_x=1, robot_location_y=1, with_symbol=False):
  for y, row in enumerate(grid):
      for x, cell in enumerate(row):
        if with_symbol:
          if robot_location_y == y and robot_location_x == x:
            symbol = "R"
          else:
            symbol = cell_to_string(cell)
          print(f"({x},{y},{symbol})", end=" ")
        else:
          print(f"({x},{y})", end=" ")
      print()


# Funktion för att uppdatera och visa rutnätet i ett widget-fönster
def render_grid(render_target, grid, robot_pose):
  with render_target:
    clear_output(wait=True)       # Rensa föregående utskrift
    print()
    print_grid(grid, robot_pose[0], robot_pose[1])  # Skriv ut rutnätet
    print()



* **Uppgift 1.0.3:** Kör kodcellen för att visualisera en genererad probleminstans.

  * Fråga 2: Hur stor är rutnätsvärlden som genereras?
  * Fråga 3: Vad händer om du ändrar wall_density (mellan 0.0 och 1.0)?
  * Fråga 4: Vad händer om du ändrar dirt_density (mellan 0.0 och 1.0)?

(Glöm inte att köra om cellerna när du ändrar dem)
  

In [ ]:
# Skapa ett scenario
# ------------------

# Scenarieparametrar (ändra gärna dessa!)
grid_size    = 10   # Rutnätets storlek (10x10 celler)
wall_density = 0.1  # Andel väggar (0.1 = 10%)
dirt_density = 0.1  # Andel smuts (0.1 = 10%)
random_home  = False  # Slumpmässig startposition?

# Generera scenariot
grid, robot_pos = generate_scenario(grid_size, wall_density, dirt_density, random_home)

# Visualisera scenariot:
# Skapa ett utdatawidget för rutnätsvisning
output_grid = widgets.Output()
render_grid(output_grid, grid, robot_pos)
print("Rutnätsvärlden (utskrift av miljön):")
display(output_grid)



---



* **Uppgift 1.0.4:** Kör kodcellen för att visualisera platsen för varje rutnätscell.

  * Fråga 5: Notera hur x ökar åt höger. Hur ändras y nedåt? Är det som förväntat?

(Det här är hur en AI-agent ser världen: (x,y)-koordinat och celltyp, för varje cell.)

In [ ]:
# Visa mer information om scenariot
print("Platsen (x,y) för varje rutnätscell:")
print_grid_indices(grid)
print()
print("Platsen för varje rutnätscell tillsammans med dess celltyp (x,y,typ):")
print_grid_indices(grid, robot_pos[0], robot_pos[1], with_symbol=True)

# Städrobot - Lab 1.1 - En simulering av en probleminstans
Rutnätet från tidigare är bara en representation av världens tillstånd.
Vi behöver lägga till en fullständig miljö som en AI-agent kan interagera med (så att världen, dvs. rutnätet, förändras baserat på agentens handlingar).

För att leka med en städrobot för hand (och för att en AI-agent ska kunna styra den) behöver vi också en slags simulering som möjliggör fram-och-tillbaka-interaktion mellan roboten och miljön.

Simuleringen håller också koll på poängen (vi vill uppnå en bra poäng genom att städa all smuts och sedan återvända hem, med så få åtgärder som möjligt).

* **Uppgift 1.1.1:** Läs snabbt igenom koden. Fokusera på funktionsnamn och beskrivningar.
  * Fråga 1: Vilka åtgärder kan städroboten utföra?

In [ ]:
# Definiera alla agentåtgärder (vad roboten kan göra)
AGENT_ACTION_NOOP        = 0  # Gör ingenting
AGENT_ACTION_MOVE_LEFT   = 1  # Rör sig vänster
AGENT_ACTION_MOVE_DOWN   = 2  # Rör sig nedåt
AGENT_ACTION_MOVE_RIGHT  = 3  # Rör sig höger
AGENT_ACTION_MOVE_UP     = 4  # Rör sig uppåt
AGENT_ACTION_SUCK_DIRT   = 5  # Dammsug smuts

# Lista med alla möjliga åtgärder
AGENT_ACTIONS = [
  AGENT_ACTION_NOOP,
  AGENT_ACTION_MOVE_UP,
  AGENT_ACTION_MOVE_DOWN,
  AGENT_ACTION_MOVE_LEFT,
  AGENT_ACTION_MOVE_RIGHT,
  AGENT_ACTION_SUCK_DIRT
]

# Endast rörelsåtgärder (allt mellan NOOP och SUCK_DIRT)
AGENT_MOVE_ACTIONS = AGENT_ACTIONS[1:-1]


# Omvandla åtgärd till text för utskrift
def action_to_string(action):
  if action == AGENT_ACTION_NOOP:
    return "NOOP"
  if action == AGENT_ACTION_MOVE_UP:
    return "UPP"
  if action == AGENT_ACTION_MOVE_DOWN:
    return "NER"
  if action == AGENT_ACTION_MOVE_LEFT:
    return "VÄNSTER"
  if action == AGENT_ACTION_MOVE_RIGHT:
    return "HÖGER"
  if action == AGENT_ACTION_SUCK_DIRT:
    return "DAMMSUG"
  return f"Fel, ingen åtgärd kallas {action}"


# ---------------------------------------------------------------------------
# Miljöklassen - representerar världen och hanterar robotens åtgärder
# Den simulerar hur världen förändras när roboten gör saker

class Environment:
  def __init__(self, grid_size, wall_density, dirt_density):
    # Skapa en ny slumpmässig värld
    self.world = generate_world(grid_size, wall_density, dirt_density)
    # Robotens startposition (övre vänstra hörnet innanför väggen)
    self.agent_location_x = 1
    self.agent_location_y = 1
    # Markera startpositionen som hem
    self.world[self.agent_location_y][self.agent_location_x] = CellType.HOME
    # Lista med meddelanden att visa i gränssnittet
    self.messages = []

  def percept(self, agent):
    # Returnera vad roboten kan uppfatta på sin nuvarande position
    return {
      "home": self.agent_location_x == 1 and self.agent_location_y == 1,  # Är roboten hemma?
      "dirt": self.world[self.agent_location_y][self.agent_location_x] == CellType.DIRT,  # Finns det smuts?
      "bump": agent.bump   # Har roboten krockat med en vägg?
    }

  def is_bump(self, x, y):
    # Kontrollera om det finns en vägg på positionen (x, y)
    return self.world[y][x] == CellType.WALL

  def execute(self, agent, action):
    # Utför en åtgärd och uppdatera miljön
    agent.bump = False
    agent.state.last_action = action

    x = self.agent_location_x  # Robotens nuvarande x-position
    y = self.agent_location_y  # Robotens nuvarande y-position

    # Beräkna agentens poäng baserat på åtgärden
    if action == AGENT_ACTION_SUCK_DIRT and self.world[y][x] == CellType.DIRT:
        agent.performance += 100.0   # +100 poäng för att städa smuts!
        self.messages.append("Poäng +100")
    else:
      agent.performance -= 1.0       # -1 poäng för varje åtgärd
      self.messages.append("Poäng -1")

    # Genomför rörelseåtgärden
    if action == AGENT_ACTION_MOVE_DOWN:
      agent.bump = self.is_bump(x, y + 1)  # Kolla om det finns en vägg nedåt
      if not agent.bump:
        self.agent_location_y += 1          # Flytta roboten nedåt
    elif action == AGENT_ACTION_MOVE_UP:
      agent.bump = self.is_bump(x, y - 1)  # Kolla om det finns en vägg uppåt
      if not agent.bump:
        self.agent_location_y -= 1          # Flytta roboten uppåt
    elif action == AGENT_ACTION_MOVE_LEFT:
      agent.bump = self.is_bump(x - 1, y)  # Kolla om det finns en vägg till vänster
      if not agent.bump:
        self.agent_location_x -= 1          # Flytta roboten vänster
    elif action == AGENT_ACTION_MOVE_RIGHT:
      agent.bump = self.is_bump(x + 1, y)  # Kolla om det finns en vägg till höger
      if not agent.bump:
        self.agent_location_x += 1          # Flytta roboten höger
    elif action == AGENT_ACTION_SUCK_DIRT:
      if self.world[y][x] == CellType.DIRT:
        self.world[y][x] = CellType.EMPTY   # Ta bort smutsen från cellen


# ---------------------------------------------------------------------------
# Agentklasser - representerar robotens "hjärna"

# Grundläggande agentens tillstånd (vad agenten vet om sig själv)
class AgentStateBase:
  def __init__(self):
    self.location_x   = 1                   # Agentens x-position
    self.location_y   = 1                   # Agentens y-position
    self.last_action  = AGENT_ACTION_NOOP   # Senaste utförda åtgärd


# En enkel agent (grundklass för alla agenter)
class Agent:
  def __init__(self):
    self.state       = AgentStateBase()  # Agentens interna tillstånd
    self.bump        = False             # Har agenten krockat med en vägg?
    self.performance = 0.0               # Agentens totalpoäng
    self.last_action = AGENT_ACTION_NOOP # Senaste åtgärd (används av simuleringen)


# En fjärrstyrd agent (styrs av användaren via knappar i gränssnittet)
class RemoteCleanerAgent(Agent):
  def execute(self, percept):
    # Den här agenten gör ingenting automatiskt - den väntar på knapptryckningar
    return AGENT_ACTION_NOOP


# ------------------------------------------------------------------------------
# Simuleringsklassen - kör simuleringen och håller koll på allt

class SimulationBase:
  def __init__(self, environment, agent, output=None, environment_output=None, render_world=True):
    self.environment       = environment       # Världen roboten lever i
    self.agent             = agent             # Agenten (roboten)
    self.step_count        = 0                 # Antal steg som tagits
    self.render_world      = render_world      # Ska världen visas?
    self.status_output     = output            # Widget för statusutskrift
    self.environment_output = environment_output  # Widget för miljövisning

    # Sätt startpoängen (straffpoäng för att inte ha städat ännu)
    self.agent.performance = -1000
    self.agent.last_action = AGENT_ACTION_NOOP
    self.environment.messages.append(f"åtgärd: {action_to_string(self.agent.last_action)}")
    self.environment.messages.append(f"position (miljö): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")

  def is_mission_complete(self):
    # Kolla om alla smuts är borta OCH roboten är hemma
    for y, row in enumerate(self.environment.world):
      for x, cell in enumerate(row):
        if cell == CellType.DIRT:
          return False  # Det finns fortfarande smuts kvar
    if self.environment.agent_location_x != 1 or self.environment.agent_location_y != 1:
      return False  # Roboten är inte hemma än
    return True  # Uppdraget är klart!

  def send_manual_command(self, manual_action):
    # Skicka ett manuellt kommando (knapptryckning) till roboten
    with output:

      # Gör ingenting om uppdraget redan är slutfört
      if self.is_mission_complete():
        print(f"Uppdrag slutfört (poäng: {self.agent.performance})")
        return

      self.step_count += 1
      print(f"{self.step_count} (Iteration)")

      percept = self.environment.percept(self.agent)  # Hämta vad roboten uppfattar
      self.agent.last_action = manual_action
      self.environment.execute(self.agent, manual_action)  # Utför åtgärden

      # Synkronisera agentens position med miljöns position
      self.agent.state.location_x = self.environment.agent_location_x
      self.agent.state.location_y = self.environment.agent_location_y

      print(f"  Åtgärd: {action_to_string(manual_action)}")
      if manual_action in AGENT_MOVE_ACTIONS:
        if self.agent.bump:
          print(f"  Misslyckades att flytta {action_to_string(manual_action)}: Vägg i vägen.")
        else:
          print(f"  Flyttade {action_to_string(manual_action)}")
      elif manual_action == AGENT_ACTION_SUCK_DIRT:
        if percept["dirt"]:
          print(f"  Städade smuts")
        else:
          print(f"  Misslyckades att städa: Ingen smuts här.")

      self.environment.messages.append(f"manuell åtgärd: {action_to_string(manual_action)}")
      self.environment.messages.append(f"position (miljö): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")
      if self.agent.bump:
        self.environment.messages.append("Krock!")
      else:
        self.environment.messages.append(" ")

      self.render_status()
      if self.environment_output:
        self.render_environment()

      if self.is_mission_complete():
        print(f"Uppdrag slutfört (poäng: {self.agent.performance})")

  # Kör ett steg i simuleringen och låt agenten fatta egna beslut
  def step(self, user_feedback=True):
    with output:
      if self.is_mission_complete():
        print(f"Uppdrag slutfört (poäng: {self.agent.performance})")
        return

      self.step_count += 1
      print(f"{self.step_count} (Iteration)")

      percept = self.environment.percept(self.agent)  # Hämta agentens perception
      action  = self.agent.execute(percept)            # Låt agenten besluta vad den ska göra
      self.environment.execute(self.agent, action)     # Utför åtgärden i miljön

      print(f"  Position: {(self.environment.agent_location_x, self.environment.agent_location_y)}   (x,y)")
      print(f"  Åtgärd: {action_to_string(action)}")
      if action in AGENT_MOVE_ACTIONS:
        if self.agent.bump:
          print(f"  Misslyckades att flytta {action_to_string(action)}: Vägg i vägen.")
        else:
          print(f"  Flyttade {action_to_string(action)}")
      elif action == AGENT_ACTION_SUCK_DIRT:
        if percept["dirt"]:
          print(f"  Städade smuts")
        else:
          print(f"  Misslyckades att städa: Ingen smuts här.")

      if user_feedback:
        self.environment.messages.append(f"åtgärd: {action_to_string(action)}")
        self.environment.messages.append(f"position (miljö): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")
        if self.agent.bump:
          self.environment.messages.append("Krock!")
        else:
          self.environment.messages.append(" ")

        self.render()

      if self.is_mission_complete():
        print(f"Uppdrag slutfört (poäng: {self.agent.performance})")

  def render_environment(self):
    # Visa miljön (världen) i widgeten
    if self.render_world:
      render_grid(self.environment_output,
                  self.environment.world,
                  (self.environment.agent_location_x, self.environment.agent_location_y))
    else:
      self.environment_output.clear_output()
    with self.environment_output:
      for line in self.environment.messages:
        print(line)
      self.environment.messages.clear()  # Rensa meddelandelistan

  def render_status(self):
    # Visa statusinformation (iteration och poäng)
    if self.status_output:
      self.status_output.clear_output()
      with self.status_output:
        print(f"Iteration: {self.step_count}")
        print(f"Poäng: {self.agent.performance}")

  def render(self):
    # Visa allt (status och miljö)
    self.render_status()
    self.render_environment()

* **Uppgift 1.1.2:** Kör kodcellerna (ovan och nedan) för att starta simuleringen.
  * Fråga 2: Vad är startpoängen?
  * Fråga 3: Vad händer med poängen om du flyttar roboten?

(Det finns knappar längst ner i gränssnittet för att skicka kommandon till roboten.)

* **Uppgift 1.1.3:** Lös tre probleminstanser genom att styra roboten med knapparna, städa upp all smuts och flytta tillbaka till robotens hemposition (startposition). En ny probleminstans skapas genom att köra om kodcellen.
  * Fråga 4: Vilken poäng fick du för de tre körningarna? Hur många åtgärder (dvs. iterationer) tog det?
  * Fråga 5: Tyckte du att någon probleminstans var svårare än en annan?

(För att slutföra en probleminstans, glöm inte att komma tillbaka till robotens hem vid (1,1) markerat med 'H' i världen.)

* **Uppgift 1.1.4:** Utforska minst tre olika probleminstanser där du ändrar väggätheten (wall_density) till högre än standardvärdet.
  * Fråga 4: Blev det svårare att lösa?

* **Uppgift 1.1.5:** Ändra väggätheten (wall_density) till noll.
  * Fråga 4: Om du låtsas vara en robot, kan du tänka dig en enkel strategi för att lösa probleminstanser utan väggar?

In [ ]:
# Sätt upp gränssnittet (GUI) och starta simulatorn!
# ---------------------------------------------------

# Skapa knappar med svenska etiketter
upp_knapp     = widgets.Button(description="Upp")
ner_knapp     = widgets.Button(description="Ner")
vänster_knapp = widgets.Button(description="Vänster")
höger_knapp   = widgets.Button(description="Höger")
dammsug_knapp = widgets.Button(description="Dammsug")

# Skapa utdatawidget för rutnätsvisning
environment_world = widgets.Output()

# Skapa utdatawidget för programstatus (poäng och iteration)
status = widgets.Output()

# Miljöparametrar  <- Ändra dessa!
grid_size    = 10   # Rutnätets storlek
wall_density = 0.1  # Andel väggar
dirt_density = 0.1  # Andel smuts

# Initiera miljö, agent och simulering
environment = Environment(grid_size, wall_density, dirt_density)
agent       = RemoteCleanerAgent()
simulation  = SimulationBase(environment, agent, output=status, environment_output=environment_world)
simulation.render()

# Rensa programutdata
output.clear_output()

# Knapp-klick-hanterare (vad som händer när man klickar på en knapp)
def klick_upp(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_UP)

def klick_ner(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_DOWN)

def klick_vänster(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_LEFT)

def klick_höger(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_RIGHT)

def klick_dammsug(b):
    simulation.send_manual_command(AGENT_ACTION_SUCK_DIRT)

upp_knapp.on_click(klick_upp)
ner_knapp.on_click(klick_ner)
vänster_knapp.on_click(klick_vänster)
höger_knapp.on_click(klick_höger)
dammsug_knapp.on_click(klick_dammsug)

# Sätt upp programutdata (höger sida av gränssnittet)
output_title = widgets.HTML(value="<b>Programutdata</b>")
view = widgets.VBox([output])
view.layout.flex_flow = 'column-reverse'
program_output = widgets.VBox([output_title, view])
program_output.layout.max_height = '400px'   # <- Bestäm höjden på utdata-widgeten
program_output.layout.width = '50%'

# Arrangera layouten
controls = widgets.HBox([upp_knapp, ner_knapp, vänster_knapp, höger_knapp, dammsug_knapp])
display(widgets.HBox([widgets.VBox([status, environment_world, controls]), program_output]))

# Städrobot - Lab 1.2 - Lös problemet i en känd miljö
Vi ska nu konstruera en AI-agent som kan lösa vilken probleminstans som helst i domänen om probleminstansen är fullt synlig för agenten från början.

Först behöver en AI-agent veta vilka åtgärder den kan ta givet ett specifikt tillstånd i världen. I domänen är det helt enkelt lika med robotens position.
> I en mer utmanande domän kan det också inkludera rotation, eller om roboten håller ett objekt i sitt grepp (och vilket objekt), om lampan är på och så vidare. För att knäcka ett ägg när man gör pannkakor kan det till exempel vara nödvändigt att först hålla ägget.



* **Uppgift 1.2.1:** Kör kodcellen för att visa en rutnätsvärld och generera alla möjliga åtgärder givet robotens nuvarande position.
  * Fråga 1: Hur många grannceller kan roboten nå från sin startposition?
  * Fråga 2: Vilken åtgärd måste roboten ta för att nå var och en av dessa grannceller?
  
* **Uppgift 1.2.2:** Ändra robotens position (new_location) till några andra positioner.
  * Fråga 1: Kan du hitta en position där roboten kan röra sig i alla fyra riktningar?
  * Fråga 2: Kan du hitta en position där roboten kan röra sig i tre riktningar?
  * Fråga 3: Kan du hitta en annan position där roboten kan röra sig i två riktningar?
  * Fråga 4: Kan du hitta en position där roboten bara kan röra sig i en riktning?
  * Fråga 5: Vad händer om du placerar roboten inuti en vägg?
  * Fråga 6: Placera roboten vid position (1,1) igen. Finns det delar av världen som roboten inte kan nå, givet att den upprepade gånger kan röra sig i sina fyra riktningar (förutom genom väggar)?

In [ ]:
# Initiera miljön med ett fast slumpfrö (ger alltid samma värld)
grid_size    = 10
wall_density = 0.3
dirt_density = 0.02
random.seed(8)  # Fast slumpfrö = samma värld varje gång
environment = Environment(grid_size, wall_density, dirt_density)


# Hitta alla giltiga grannar till en position i världen
# (dvs. de positioner man kan flytta till - inte väggar)
def get_neighbors(pos, world):
  grannar = []  # Tom lista för att samla grannar
  # Kolla alla fyra riktningar: upp, höger, ner, vänster
  # dx, dy är förändringen i x respektive y
  for dx, dy in [(0, -1), (1, 0), (0, 1), (-1, 0)]:
    x = pos[0] + dx  # Ny x-koordinat
    y = pos[1] + dy  # Ny y-koordinat
    # Kontrollera att vi är inom rutnätets gränser och att det inte är en vägg
    # len(world[0]) = bredden, len(world) = höjden
    if 0 <= x < len(world[0]) and 0 <= y < len(world) and world[y][x] != CellType.WALL:
      grannar.append((x, y))  # Lägg till som giltig granne
  return grannar


# Exempel
new_location = (1, 1)  # <----- Ändra dessa! (x,y)
print_grid(environment.world,
           robot_location_x=new_location[0],
           robot_location_y=new_location[1])
print("")
print(f"Fria grannar till {new_location}: {get_neighbors(new_location, environment.world)}   (x, y)")


Ett sätt att lösa problemet är att traversera hela rutnätsvärlden, besöka varje position minst en gång, helst med så få åtgärder som möjligt. Om roboten befinner sig på en cell med smuts, städar den. Slutligen återvänder den hem (antingen samma väg tillbaka eller via en kortare stig).

Sökutrymmet är en graf, dvs. en position (med/utan smuts) är en nod där roboten befinner sig, och varje giltig rörelseåtgärd tar den till en annan nod (annan position). Vi uppnår graftraversering med djupet-först-traversering.


* **Uppgift 1.2.3:** Kör kodcellen.

In [ ]:
# Djupet-först-traversering
# Hittar en stig för att besöka alla nåbara noder från en startnod
def dft(start_node, map):
  visited  = []           # Lista med besökta positioner
  frontier = [start_node] # Lista med positioner att besöka härnäst

  path = []  # Stigen (ordningen vi besöker noderna)
  while frontier:
    last_node = frontier.pop(-1)  # Ta den sista noden från listan (stack)
    if last_node not in visited:
      path.append(last_node)      # Lägg till i stigen
      visited.append(last_node)   # Markera som besökt

      neighbors = get_neighbors(last_node, map)  # Hitta grannar
      for neighbor in neighbors:
        frontier.append(neighbor)  # Lägg till grannar att besöka

  return path


# ---------------------------
# Söktillstånd - håller koll på en position och vägen dit
class SearchState:
  def __init__(self, location_x, location_y, parent=None, last_action=None):
    self.location_x  = location_x   # X-position
    self.location_y  = location_y   # Y-position
    self.last_action = last_action  # Åtgärden som tog oss hit
    self.parent      = parent       # Föregående tillstånd (för att rekonstruera stigen)

  def __hash__(self):
    # Används för att snabbt jämföra tillstånd - bara positionen räknas
    return hash((self.location_x, self.location_y))

  def __eq__(self, other):
    # Två tillstånd är lika om de har samma position
    return self.location_x == other.location_x and self.location_y == other.location_y


# Vad leder en åtgärd till givet ett nuvarande söktillstånd?
def apply_action(action, search_state):
  # Skapa ett nytt söktillstånd med uppdaterad position
  new_state = SearchState(search_state.location_x, search_state.location_y)
  if action == AGENT_ACTION_MOVE_UP:
    new_state.location_y -= 1   # Flytta uppåt (minska y)
  if action == AGENT_ACTION_MOVE_DOWN:
    new_state.location_y += 1   # Flytta nedåt (öka y)
  if action == AGENT_ACTION_MOVE_LEFT:
    new_state.location_x -= 1   # Flytta vänster (minska x)
  if action == AGENT_ACTION_MOVE_RIGHT:
    new_state.location_x += 1   # Flytta höger (öka x)
  return new_state


# Bredden-först-sökning (BFS - Breadth-First Search)
# Hittar den kortaste vägen (i antal åtgärder) från ett starttillstånd
# till ett tillstånd som uppfyller målvillkoret (goal_condition)
def bfs(start_state, world, goal_condition):
  visited  = []              # Besökta tillstånd
  frontier = [start_state]   # Tillstånd att utforska (kö)

  plan = []  # Handlingsplanen (listan av åtgärder)
  while frontier:
    current_state = frontier.pop(0)  # Ta det första tillståndet (kö = FIFO)
    if current_state not in visited:
      visited.append(current_state)

    if goal_condition(current_state, world):
      # Vi hittade målet! Bygg upp planen bakifrån längs föräldralänkarna
      while current_state.parent:
        plan.insert(0, current_state.last_action)
        current_state = current_state.parent
      return plan

    # Utvidga det nuvarande tillståndet (prova alla möjliga rörelser)
    for action in AGENT_MOVE_ACTIONS:
      new_state = apply_action(action, current_state)
      x = new_state.location_x
      y = new_state.location_y
      if world[y][x] != CellType.WALL and new_state not in visited:
        new_state.parent      = current_state  # Spara föräldralänken
        new_state.last_action = action         # Spara åtgärden som tog oss hit
        frontier.append(new_state)

  # Sökningen är klar men ingen plan hittades
  return []


* **Uppgift 1.2.4:** Kör kodcellen för att skapa en miljö och hitta en djupet-först-traverseringsstig.
  * Fråga 1: Täcker den alla nåbara positioner?
  * Fråga 2: Var slutar den?
  * Fråga 3: Kan roboten följa stigen med sina åtgärder (en åtgärd per steg i stigen)?
  
(Förslag: Använd ett rutnätspapper och rita djupet-först-traverseringsstigen för hand)

In [ ]:
grid_size    = 6
dirt_density = 0.08
environment  = Environment(grid_size, 0.1, dirt_density)

print_grid(environment.world, robot_location_x=1, robot_location_y=1)

path = dft((1, 1), environment.world)
print("Djupet-först-traverseringsstig:", path)
print("Stiglängd:", len(path) - 1)  # Räkna inte startpositionen



---


* **Uppgift 1.2.5:** Kör kodcellen för att hitta en plan (sekvens av åtgärder) för varje del av stigen. Tillsammans bildar alla planer en lång plan.
  * Fråga 1: Kan roboten följa stigen nu med hjälp av sin plan?
  * Fråga 2: Hur mycket längre är planen (i antal åtgärder) jämfört med djupet-först-traverseringsstigen (i antal koordinater)?
  
(Förslag: Använd ett rutnätspapper och rita en linje som följer planen)

In [ ]:
plan = []
for n in range(1, len(path)):
  # Hitta kortaste vägen från föregående position till nästa position i stigen
  p = bfs(SearchState(path[n-1][0], path[n-1][1]),
          environment.world,
          lambda state, world: state.location_x == path[n][0] and state.location_y == path[n][1])
  plan.extend(p)  # Lägg till delplanen i den totala planen

print("")
print("Plan för att besöka alla positioner i stigen ovan (i ordning)")
print("--------------------------------------------------------------")
print("plan (heltal):", plan)
print("plan (text):", [action_to_string(action) for action in plan])
print("planlängd:", len(plan))



---


En bättre strategi, eftersom vi vet (eller enkelt kan ta reda på) var smutsen finns, är att hitta den närmaste smutsen och ta sig dit med så få åtgärder som möjligt. Vi använder bredden-först-sökning för att hitta en stig till den första smutspositionen vi hittar under sökningen.


* **Uppgift 1.2.7:** Kör kodcellen för att hitta en plan till den närmaste smutspositionen.
  * Fråga 1: Om du följer planen från robotens startposition (1,1), var hamnar du?
  * Fråga 2: Vad är den kortaste planen till nästa smuts?
  * Fråga 3: Kör om ett par gånger med olika robotpositioner: Tar planerna roboten till den närmaste smutsen?

In [ ]:
robot_location = (1, 1)  # <----- Ändra dessa! (x,y)

print_grid(environment.world, robot_location_x=robot_location[0], robot_location_y=robot_location[1])

# Hitta kortaste vägen till närmaste smuts med bredden-först-sökning
plan = bfs(SearchState(robot_location[0], robot_location[1]),
           environment.world,
           lambda state, world: world[state.location_y][state.location_x] == CellType.DIRT)
print(f"plan = [{[action_to_string(action) for action in plan]}]")



---

Att upprepa detta beteende låter robot-AI:n effektivt hitta alla smutspunkter. När all smuts är städad kan den på liknande sätt hitta den kortaste sekvensen av åtgärder som tar den hem för att slutföra probleminstansen.


* **Uppgift 1.2.8:** Kör kodcellerna nedan.

In [ ]:
# Städagent med inre tillstånd och känd karta
# ---------------------------------------------

# Utökad simulering som även visar agentens interna karta
class Simulation(SimulationBase):
  def __init__(self, environment, agent, output=None, environment_output=None,
               agent_world_output=None, render_world=True):
    # Initiera bassimuleringsklassen
    SimulationBase.__init__(self, environment, agent, output, environment_output, render_world)
    self.agent_world_output = agent_world_output  # Widget för agentens interna karta
    self.environment.messages.append(
      f"position (agent): ({self.agent.state.location_x}, {self.agent.state.location_y})")

  def render_environment(self):
    # Visa agentens interna karta (vad agenten tror om världen)
    render_grid(self.agent_world_output,
                self.agent.state.world,
                (self.agent.state.location_x, self.agent.state.location_y))
    # Visa den verkliga miljön
    SimulationBase.render_environment(self)

  def send_manual_command(self, manual_action):
    # Skicka manuellt kommando och uppdatera agentens tillstånd efteråt
    SimulationBase.send_manual_command(self, manual_action)
    self.agent.state.last_action = AGENT_ACTION_NOOP
    self.agent.state.update_location(self.environment.percept(self.agent))

    self.environment.messages.append(f"manuell åtgärd: {action_to_string(manual_action)}")
    self.environment.messages.append(
      f"position (miljö): ({self.environment.agent_location_x}, {self.environment.agent_location_y})")
    self.environment.messages.append(
      f"position (agent): ({self.agent.state.location_x}, {self.agent.state.location_y})")
    if self.agent.bump:
      self.environment.messages.append("Krock!")
    else:
      self.environment.messages.append(" ")

    SimulationBase.render(self)


# Agentens tillstånd - håller koll på agentens interna karta och position
class AgentState(AgentStateBase):
  def __init__(self, world_max_width, world_max_height):
    # Grundläggande position och senaste åtgärd
    self.location_x  = 1
    self.location_y  = 1
    self.last_action = AGENT_ACTION_NOOP

    # Agentens interna karta - till en början är allt okänt
    self.world_width  = world_max_width
    self.world_height = world_max_height
    self.world = [[CellType.UNKNOWN for _ in range(world_max_width)]
                  for _ in range(world_max_height)]
    self.world[1][1] = CellType.HOME  # Startpositionen är känd som hem

  def update_location(self, percept):
    # Uppdatera agentens position och karta baserat på vad den uppfattar
    with output:

      # Om det var en krock, markera väggen i kartan
      if percept["bump"]:
        if self.last_action == AGENT_ACTION_MOVE_UP:
          self.update_world(self.location_x, self.location_y - 1, CellType.WALL)
        elif self.last_action == AGENT_ACTION_MOVE_DOWN:
          self.update_world(self.location_x, self.location_y + 1, CellType.WALL)
        elif self.last_action == AGENT_ACTION_MOVE_LEFT:
          self.update_world(self.location_x - 1, self.location_y, CellType.WALL)
        elif self.last_action == AGENT_ACTION_MOVE_RIGHT:
          self.update_world(self.location_x + 1, self.location_y, CellType.WALL)

      # Uppdatera positionen om det inte var en krock
      if not percept["bump"]:
        if self.last_action == AGENT_ACTION_MOVE_UP:
          self.location_y -= 1
        elif self.last_action == AGENT_ACTION_MOVE_DOWN:
          self.location_y += 1
        elif self.last_action == AGENT_ACTION_MOVE_LEFT:
          self.location_x -= 1
        elif self.last_action == AGENT_ACTION_MOVE_RIGHT:
          self.location_x += 1

      # Uppdatera kartan baserat på vad roboten ser på sin nuvarande position
      if percept["dirt"]:
        if self.last_action == AGENT_ACTION_SUCK_DIRT:
          self.update_world(self.location_x, self.location_y, CellType.EMPTY)  # Smutsen är borta
        else:
          self.update_world(self.location_x, self.location_y, CellType.DIRT)   # Det finns smuts här
      elif percept["home"]:
        self.update_world(self.location_x, self.location_y, CellType.HOME)     # Det här är hemmet
      else:
        self.update_world(self.location_x, self.location_y, CellType.EMPTY)    # Tom cell

  def update_world(self, x, y, info):
    # Uppdatera en cell i agentens interna karta
    self.world[y][x] = info

  def print_world_debug(self):
    # Skriv ut agentens karta (för felsökning)
    print_grid(self.world, self.location_x, self.location_y)
    print()

In [ ]:
# Slumpmässig städagent - väljer slumpmässiga åtgärder
class RandomCleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)         # Initiera grundläggande agentegenskaper
    self.state = AgentState(10, 10)  # Agentens interna karta (max 10x10)

  def execute(self, percept):
    # Uppdatera kartan och välj en slumpmässig åtgärd
    self.state.update_location(percept)
    return random.choice(AGENT_ACTIONS)  # Välj slumpmässig åtgärd


# ------------------------------------------------------------------------------
# Plats för en framtida BFS-planeringsagent med bredden-först-traversering
class BftPlanningCleanerAgent(Agent):
  pass  # Inte implementerad ännu


# ------------------------------------------------------------------------------
# BFS-planeringsagent - hittar kortaste vägen till närmaste smuts
class BfsPlanningCleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)          # Initiera grundläggande agentegenskaper
    self.state = AgentState(10, 10)  # Agentens interna karta
    self.plan  = []               # Agentens nuvarande plan (lista av åtgärder)

  def execute(self, percept):
    # Uppdatera agentens tillstånd baserat på vad den uppfattar
    self.state.update_location(percept)

    with output:

      print(f"  Agent utför")
      print_grid(self.state.world, self.state.location_x, self.state.location_y, margin="    ")
      print(f"    Position: {(self.state.location_x, self.state.location_y)}   (x,y)")
      print(f"    Percept: {percept}")

      # Om det finns smuts här - städa genast!
      if percept["dirt"]:
        print(f"    Utför åtgärd: {action_to_string(AGENT_ACTION_SUCK_DIRT)}")
        return AGENT_ACTION_SUCK_DIRT

      # Om vi inte har en plan - hitta vägen till närmaste smuts
      if not self.plan:
        self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                        self.state.world,
                        lambda state, world: world[state.location_y][state.location_x] == CellType.DIRT)
        print(f"    ny plan = [{[action_to_string(a) for a in self.plan]}]")

      if not self.plan:
        # All tillgänglig smuts är städad
        if self.state.world[self.state.location_y][self.state.location_x] != CellType.HOME:
          # Hitta vägen hem
          print("    Hittar vägen hem")
          self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                          self.state.world,
                          lambda state, world: world[state.location_y][state.location_x] == CellType.HOME)
          print(f"    plan = [{[action_to_string(a) for a in self.plan]}]")
        else:
          # Städningen är klar!
          print("    Städning klar!")
          return AGENT_ACTION_NOOP

      if self.plan:
        # Utför den första åtgärden i planen (och ta bort den från planen)
        action = self.plan.pop(0)
        print(f"    Kvarvarande plan: [{[action_to_string(a) for a in self.plan]}]")
        print(f"    Utför åtgärd: {action_to_string(action)}")
        return action

    # Vi bör aldrig hamna här
    return AGENT_ACTION_NOOP


* **Uppgift 1.2.9:** Kör kodcellen nedan för att starta gränssnittet.
  * Fråga 1: Testa den slumpmässiga agenten. Använd "Steg" för att köra simuleringen steg för steg (eller Steg10, Steg100 för fler steg). Verkar det vara lätt eller svårt för den slumpmässiga agenten att lösa problemet?
  * Fråga 2: Testa BFS-planeringsagenten. På vilket sätt är den här agenten bättre än den slumpmässiga agenten? Vad gör den som är mer intelligent jämfört med den slumpmässiga agenten?

In [ ]:
# Skapa gränssnittet för simuleringsprogrammet
# ---------------------------------------------

# Skapa knappar
steg_knapp     = widgets.Button(description="Steg")
steg_10        = widgets.Button(description="Steg10")
steg_100       = widgets.Button(description="Steg100")
rensa          = widgets.Button(description="Rensa utdata")
återställ_knapp = widgets.Button(description="Återställ")
ny_knapp       = widgets.Button(description="Ny")

# Agentsalternativ för rullgardinsmenyn
agenter = [
    ('Slumpmässig', 'random'),
    ('BFS-planering', 'bfs planning')
]

# Rullgardinsmeny för att välja agent
agent_dropdown = widgets.Dropdown(
    options=agenter,
    value='bfs planning',
    description='Robot',
)

# Utdatawidgets
environment_world = widgets.Output()  # Miljöns verkliga värld
agent_world       = widgets.Output()  # Agentens interna karta

# Dummy-utdata (används som avstånd i layouten)
dummy_output = widgets.Output()
with dummy_output:
  print(" ")

# Statusutdata
status = widgets.Output()

# Spara slumpstatus för att kunna återskapa samma värld
random_state = random.getstate()


def ny_miljö():
  # Skapa en ny slumpmässig miljö
  global random_state
  random.random()                  # Hoppa framåt i slumpsekvensen
  random_state = random.getstate()  # Spara det nya slumptillståndet
  återställ(agent_dropdown.value)


def återställ(agent_typ):
  # Initiera miljö, agent och simulering
  global environment, agent, simulation
  output.clear_output()           # Rensa loggutdata
  random.setstate(random_state)   # Återställ slumptillståndet

  # Miljöparametrar
  grid_size    = 6
  dirt_density = 0.1
  wall_density = 0.05

  environment = Environment(grid_size, wall_density, dirt_density)
  if agent_typ == 'random':
    agent = RandomCleanerAgent()
  if agent_typ == 'bfs planning':
    agent = BfsPlanningCleanerAgent()

  # Ge agenten en kopia av den kända miljön (rad för rad)
  agent.state.world = [rad[:] for rad in environment.world]

  simulation = Simulation(environment, agent, output=status,
                          environment_output=environment_world,
                          agent_world_output=agent_world)
  simulation.render()


# Initiera den första miljön
återställ(agent_dropdown.value)


from time import sleep

def kör_steg(b, N, increment=1):
  # Kör N steg i simuleringen
  b.disabled = True   # Inaktivera knappen under körning
  for n in range(N):
    if simulation.is_mission_complete():
      simulation.render()
      break
    if n % increment == 0:
      simulation.step()          # Kör ett steg och visa resultatet
      sleep(0.5)                 # Liten paus för animationseffekt
    else:
      simulation.environment.messages.clear()
      simulation.step(user_feedback=False)  # Kör utan att visa
  b.disabled = False  # Aktivera knappen igen


# Knapp-klick-hanterare (namngivna funktioner i stället för lambda)
def klick_ny(b):
    ny_miljö()

def klick_återställ(b):
    återställ(agent_dropdown.value)

def agent_ändrad(b):
    if b['name'] == 'value' and b['type'] == 'change':
        återställ(b['new'])

def klick_steg(b):
    simulation.step()

def klick_steg_100(b):
    kör_steg(b, 100, increment=10)

def klick_steg_10(b):
    kör_steg(b, 10)

def klick_rensa(b):
    output.clear_output()


ny_knapp.on_click(klick_ny)
återställ_knapp.on_click(klick_återställ)
agent_dropdown.observe(agent_ändrad)
steg_knapp.on_click(klick_steg)
steg_100.on_click(klick_steg_100)
steg_10.on_click(klick_steg_10)
rensa.on_click(klick_rensa)

# Arrangera layouten
controls = widgets.HBox([steg_knapp, steg_10, steg_100, rensa])
display(widgets.HBox([
    widgets.VBox([
        widgets.HBox([ny_knapp, återställ_knapp, agent_dropdown, dummy_output]),
        status,
        widgets.HBox([environment_world, dummy_output, agent_world]),
        controls
    ]),
    program_output
]))

# Städrobot - Lab 1.3 - Byt till en okänd miljö

Här gör vi världen okänd så att roboten måste utforska världen för att hitta var smutsen (och väggarna) finns.

* **Uppgift 1.3.1:** Kör kodcellen nedan för att starta gränssnittet. Lös problemet tre gånger (kör om kodcellerna).
  * Fråga 1: Är det här problemet svårare än det där världen är synlig (känd) från början?
  * Fråga 2: Kan sådana probleminstanser lösas med vår tidigare effektiva lösning (BFS till närmaste smuts)?
  * Fråga 3: Vad är en bra alternativ strategi?

In [ ]:
# En fjärrstyrd agent med sin egen okända världskarta
class CleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)          # Initiera grundläggande agentegenskaper
    self.state = AgentState(10, 10)  # Agentens interna karta (börjar okänd)

  def execute(self, percept):
    # Uppdatera kartan och gör ingenting (väntar på manuella kommandon)
    self.state.update_location(percept)
    return AGENT_ACTION_NOOP


# Skapa knappar med svenska etiketter
upp_knapp     = widgets.Button(description="Upp")
ner_knapp     = widgets.Button(description="Ner")
vänster_knapp = widgets.Button(description="Vänster")
höger_knapp   = widgets.Button(description="Höger")
dammsug_knapp = widgets.Button(description="Dammsug")

# Utdatawidgets
environment_world = widgets.Output()  # Miljöns verkliga värld
agent_world       = widgets.Output()  # Agentens interna karta

# Dummy-utdata
dummy_output = widgets.Output()
with dummy_output:
  print(" ")

# Statusutdata
status = widgets.Output()

# Initiera miljö, agent och simulering
environment = Environment(grid_size, wall_density, dirt_density)
agent       = CleanerAgent()
simulation  = Simulation(environment, agent, output=status,
                         environment_output=environment_world,
                         agent_world_output=agent_world,
                         render_world=False)
simulation.render()

# Rensa programutdata
output.clear_output()

# Knapp-klick-hanterare
def klick_upp(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_UP)

def klick_ner(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_DOWN)

def klick_vänster(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_LEFT)

def klick_höger(b):
    simulation.send_manual_command(AGENT_ACTION_MOVE_RIGHT)

def klick_dammsug(b):
    simulation.send_manual_command(AGENT_ACTION_SUCK_DIRT)

upp_knapp.on_click(klick_upp)
ner_knapp.on_click(klick_ner)
vänster_knapp.on_click(klick_vänster)
höger_knapp.on_click(klick_höger)
dammsug_knapp.on_click(klick_dammsug)

# Arrangera layouten
controls = widgets.HBox([upp_knapp, ner_knapp, vänster_knapp, höger_knapp, dammsug_knapp])
display(widgets.HBox([widgets.VBox([status, widgets.HBox([agent_world]), controls]), program_output]))

# Städrobot - Lab 1.4 - Lös problemet i en okänd miljö

Vi ändrar BFS för att leta efter den närmaste **okända** positionen, i stället för den närmaste smutsen. Nu har vi en AI-agent som återigen är effektiv för den här nya svårare domänen.

* **Uppgift 1.4.1:** Kör kodcellen nedan för att starta gränssnittet.
  * Fråga 1: Använd den slumpmässiga agenten. Använd 'Steg' för att stega simuleringen ett steg i taget (eller använd 'Steg10' eller 'Steg100' för snabbare simulering). Verkar det vara lätt eller svårt för den slumpmässiga agenten att lösa problemet? Varför tror du det?
  * Fråga 2: Använd BFS-planeringsagenten (välj den i rullgardinsmenyn). På vilket sätt är den här agenten bättre än den slumpmässiga agenten? Vad gör den mer intelligent?
  * Fråga 3: Prova en större värld (t.ex. grid_size=10). Hur många drag tar det för BFS-planeringsagenten att lösa det? Verkar det vara svårare för den slumpmässiga agenten?
  * Fråga 4: Reflektera över skillnaden mellan den här AI-agenten och de tidigare.

In [ ]:
# BFS-planeringsagent för okänd miljö - utforskar okända positioner
class BfsPlanningUnknownCleanerAgent(Agent):
  def __init__(self):
    Agent.__init__(self)          # Initiera grundläggande agentegenskaper
    self.state = AgentState(10, 10)  # Agentens interna karta
    self.plan  = []               # Agentens nuvarande plan

  def execute(self, percept):
    # Uppdatera agentens tillstånd baserat på vad den uppfattar
    self.state.update_location(percept)

    with output:

      print(f"  Agent utför")
      print_grid(self.state.world, self.state.location_x, self.state.location_y, margin="    ")
      print(f"    Position: {(self.state.location_x, self.state.location_y)}   (x,y)")
      print(f"    Percept: {percept}")

      # Om det finns smuts här - städa genast!
      if percept["dirt"]:
        print(f"    Utför åtgärd: {action_to_string(AGENT_ACTION_SUCK_DIRT)}")
        return AGENT_ACTION_SUCK_DIRT

      # NYTT: Leta efter okänd cell för att utforska - inte smuts att städa
      if not self.plan:
        self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                        self.state.world,
                        lambda state, world: world[state.location_y][state.location_x] == CellType.UNKNOWN)
        print(f"    ny plan = [{[action_to_string(a) for a in self.plan]}]")

      if not self.plan:
        # All tillgänglig smuts är städad
        if self.state.world[self.state.location_y][self.state.location_x] != CellType.HOME:
          # Hitta vägen hem
          print("    Hittar vägen hem")
          self.plan = bfs(SearchState(self.state.location_x, self.state.location_y),
                          self.state.world,
                          lambda state, world: world[state.location_y][state.location_x] == CellType.HOME)
          print(f"    plan = [{[action_to_string(a) for a in self.plan]}]")
        else:
          # Städningen är klar!
          print("    Städning klar!")
          return AGENT_ACTION_NOOP

      if self.plan:
        # Utför den första åtgärden i planen (och ta bort den från planen)
        action = self.plan.pop(0)
        print(f"    Kvarvarande plan: [{[action_to_string(a) for a in self.plan]}]")
        print(f"    Utför åtgärd: {action_to_string(action)}")
        return action

    # Vi bör aldrig hamna här
    return AGENT_ACTION_NOOP


# Skapa knappar
steg_knapp      = widgets.Button(description="Steg")
steg_10         = widgets.Button(description="Steg10")
steg_100        = widgets.Button(description="Steg100")
rensa           = widgets.Button(description="Rensa utdata")
återställ_knapp = widgets.Button(description="Återställ")
ny_knapp        = widgets.Button(description="Ny")

# Agentsalternativ
agenter = [
    ('Slumpmässig', 'random'),
    ('BFS-planering', 'bfs planning')
]

# Rullgardinsmeny
agent_dropdown = widgets.Dropdown(
    options=agenter,
    value='random',
    description='Robot',
)

# Utdatawidgets
environment_world = widgets.Output()
agent_world       = widgets.Output()

# Dummy-utdata
dummy_output = widgets.Output()
with dummy_output:
  print(" ")

# Statusutdata
status = widgets.Output()

# Spara slumpstatus
random_state = random.getstate()


def ny_miljö():
  # Skapa en ny slumpmässig miljö
  global random_state
  random.random()
  random_state = random.getstate()
  återställ(agent_dropdown.value)


def återställ(agent_typ):
  # Initiera miljö, agent och simulering
  global environment, agent, simulation
  output.clear_output()
  random.setstate(random_state)
  grid_size    = 5          # <---------------------------------- Ändra det här värdet!
  dirt_density = 0.1
  wall_density = 0.05       # <---------------------------------- Ändra det här värdet!
  environment = Environment(grid_size, wall_density, dirt_density)
  if agent_typ == 'random':
    agent = RandomCleanerAgent()
  if agent_typ == 'bfs planning':
    agent = BfsPlanningUnknownCleanerAgent()
  simulation = Simulation(environment, agent, output=status,
                          environment_output=environment_world,
                          agent_world_output=agent_world)
  simulation.render()


# Initiera den första miljön
återställ(agent_dropdown.value)


from time import sleep

def kör_steg(b, N, increment=1):
  b.disabled = True
  for n in range(N):
    if simulation.is_mission_complete():
      simulation.render()
      break
    if n % increment == 0:
      simulation.step()
      sleep(0.5)
    else:
      simulation.environment.messages = []
      simulation.step(user_feedback=False)
  b.disabled = False


# Knapp-klick-hanterare
def klick_ny(b):
    ny_miljö()

def klick_återställ(b):
    återställ(agent_dropdown.value)

def agent_ändrad(b):
    if b['name'] == 'value' and b['type'] == 'change':
        återställ(b['new'])

def klick_steg(b):
    simulation.step()

def klick_steg_100(b):
    kör_steg(b, 100, increment=10)

def klick_steg_10(b):
    kör_steg(b, 10)

def klick_rensa(b):
    output.clear_output()


ny_knapp.on_click(klick_ny)
återställ_knapp.on_click(klick_återställ)
agent_dropdown.observe(agent_ändrad)
steg_knapp.on_click(klick_steg)
steg_100.on_click(klick_steg_100)
steg_10.on_click(klick_steg_10)
rensa.on_click(klick_rensa)

# Arrangera layouten
controls = widgets.HBox([steg_knapp, steg_10, steg_100, rensa])
display(widgets.HBox([
    widgets.VBox([
        widgets.HBox([ny_knapp, återställ_knapp, agent_dropdown, dummy_output]),
        status,
        widgets.HBox([environment_world, dummy_output, agent_world]),
        controls
    ]),
    program_output
]))